# A Glance at the Dataset and the Tokenizers

*Chapter 4 - Multilingual Named Entity Recognition · Natural Language Processing with Transformers (Tunstall, von Werra and Wolf)*

## Overview

This is the first notebook of the Chapter 4 project. The goal of the chapter is to build a **named entity recognition (NER)** model that works in several languages. Before we train anything, we need to understand two things: the **data** and the **tokenizers**. This notebook looks at both.

NER is a *token classification* task. The model reads a sentence and gives every word a label, for example "this word is part of a person name". Here we look for three entity types: persons (`PER`), organizations (`ORG`) and locations (`LOC`).

## Objective

- Find the NER part of the XTREME benchmark (called **PAN-X**).
- Build a small multilingual corpus with German, French, Italian and English.
- Read the data format: words, integer tags and readable tag names.
- Count how many entities of each type exist in the German corpus.
- Compare the tokenizer of BERT with the tokenizer of XLM-RoBERTa (XLM-R).

## Workflow

| Step | What happens | Section |
|---|---|---|
| 1 | List the XTREME configurations and keep the PAN-X ones | Section 1 |
| 2 | Load four languages and keep a share of each, based on how widely each language is spoken | Section 2 |
| 3 | Inspect one example, the features and the label names | Section 3 |
| 4 | Count the entities in every German split | Section 3 |
| 5 | Tokenize one sentence with two tokenizers and compare the results | Section 4 |

## Tools and data

- **`datasets`** loads XTREME from the Hugging Face Hub and gives us `Dataset` and `DatasetDict` objects.
- **`pandas`** shows small tables, which are easier to read than raw Python lists.
- **`transformers`** provides `AutoTokenizer` for both tokenizers.
- **Data:** the PAN-X subset of XTREME. It contains sentences from Wikipedia with entity tags in the IOB2 format (explained in Section 3).
- **Tokenizers:** `bert-base-cased` and `xlm-roberta-base`.

## Notes for the reader

- The notebooks of this chapter share objects with each other through `import_ipynb`. When another notebook imports this one, **all of its cells run**. For this reason, this notebook only contains data loading and inspection, and no training.
- The line `os.environ["USE_TF"] = "0"` tells `transformers` not to load TensorFlow. The project uses PyTorch.
- Internet access is needed the first time, because the dataset and the tokenizers are downloaded.

## Section 1 - Exploring the XTREME Benchmark

XTREME is a benchmark that measures how well a model transfers what it learns in one language to other languages (Hu et al., 2020). It has several tasks. On the Hugging Face Hub, every combination of task and language is one **configuration**. We can list all configuration names with `get_dataset_config_names("xtreme")` without downloading the data.

The NER task in XTREME is called **PAN-X**, and each of its configurations starts with `PAN`. The first cell lists all names, and the second cell keeps only the PAN-X ones. The first three are `PAN-X.af`, `PAN-X.ar` and `PAN-X.bg` (Afrikaans, Arabic and Bulgarian), so the language code is the part after the dot.

> **Note:** The first cell uses `%%capture`, so the printed number of configurations is hidden and is not saved in the notebook.

In [2]:
%%capture
import os
os.environ["USE_TF"] = "0"

from datasets import get_dataset_config_names

xtreme_subsets = get_dataset_config_names("xtreme")
print(f"XTREME has {len(xtreme_subsets)} configuations")

In [2]:
panx_subsets = [s for s in xtreme_subsets if s.startswith("PAN")]
panx_subsets[:3]

['PAN-X.af', 'PAN-X.ar', 'PAN-X.bg']

## Section 2 - Building a Multilingual Corpus

### 2.1 Loading and Downsampling Four Languages

The chapter follows one practical story: a company that works in a country where German, French, Italian and English are all used. German is spoken most, and English least. We copy this situation by keeping a different share of each language corpus. The shares are written in `fracs`: German 62.9%, French 22.9%, Italian 8.4% and English 5.9%.

For each language, the code does the following:

1. Load all splits (`train`, `validation`, `test`) with `load_dataset("xtreme", name=f"PAN-X.{lang}")`.
2. Shuffle every split with `seed=0`, so the sample is random but can be repeated.
3. Keep the first `frac × num_rows` rows.

The result is stored in `panx_ch`. This is a `defaultdict(DatasetDict)`, so a new language key creates an empty `DatasetDict` automatically, and we can write `panx_ch["de"]["train"]`. Later notebooks import `panx_ch`.

In [4]:
from collections import defaultdict
from datasets import DatasetDict
from datasets import load_dataset

# Four languages and the share of each corpus that we keep
langs = ["de", "fr", "it", "en"]
fracs = [0.629, 0.229, 0.084, 0.059]
# Return a DatasetDict if a key doesn't exist
panx_ch = defaultdict(DatasetDict)
for lang, frac in zip(langs, fracs):
    # Load monolingual corpus
    ds = load_dataset("xtreme", name=f"PAN-X.{lang}")
    # Shuffle and downsample each split according to spoken proportion
    for split in ds:
        panx_ch[lang][split] = (
            ds[split]
            .shuffle(seed=0)
            .select(range(int(frac * ds[split].num_rows)))
        )

### 2.2 Size of the Training Sets

The next cell counts the training rows of each language. The numbers show a strong imbalance: German has more than ten times as many training examples as English. This matters later, when we test how well a model trained on one language works on another.

| Language | Share kept | Training examples |
|---|---|---|
| `de` | 0.629 | 12,580 |
| `fr` | 0.229 | 4,580 |
| `it` | 0.084 | 1,680 |
| `en` | 0.059 | 1,180 |

Each share multiplied by 20,000 gives the number in the table (for example, 0.629 × 20,000 = 12,580). This means that the full PAN-X training split of each language has 20,000 rows. We derived this from the printed numbers and did not read it from the dataset.

In [5]:
import pandas as pd
pd.DataFrame({lang: [panx_ch[lang]["train"].num_rows] for lang in langs})

,de,fr,it,en
0,12580,4580,1680,1180


## Section 3 - Understanding the Data Format

### 3.1 One Training Example

Every example has three fields:

| Field | Meaning |
|---|---|
| `tokens` | The sentence as a list of words |
| `ner_tags` | One integer class id for each word |
| `langs` | The language code, repeated for each word |

The tags use the **IOB2** scheme. `B-` marks the *beginning* of an entity, `I-` marks a word *inside* the same entity, and `O` means the word is *outside* any entity.

| Id | Tag | Meaning |
|---|---|---|
| 0 | `O` | Not part of an entity |
| 1 | `B-PER` | Start of a person name |
| 2 | `I-PER` | Inside a person name |
| 3 | `B-ORG` | Start of an organization name |
| 4 | `I-ORG` | Inside an organization name |
| 5 | `B-LOC` | Start of a location name |
| 6 | `I-LOC` | Inside a location name |

In the first German example, "Danziger Bucht" has the ids `5, 6`, so it is one location that has two words.

In [6]:
element = panx_ch["de"]["train"][0]
for k, v in element.items():
    print(f"{k} --> {v}")

tokens --> ['2.000', 'Einwohnern', 'an', 'der', 'Danziger', 'Bucht', 'in', 'der', 'polnischen', 'Woiwodschaft', 'Pommern', '.']
ner_tags --> [0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0]
langs --> ['de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de']


### 3.2 Features and Label Names

The `features` attribute describes the type of each column. Here, `ner_tags` is a list of `ClassLabel` values, and this object knows the names of the seven classes. The next cell takes the inner `ClassLabel` with `.feature` and stores it in `tags`. Later notebooks use it many times: `tags.names` (the list of names), `tags.int2str` (id to name) and `tags.num_classes` (the number of classes, which is 7).

In [7]:
for k, v in panx_ch["de"]["train"].features.items():
    print(f"{k} --> {v}")

tokens --> List(Value('string'))
ner_tags --> List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))
langs --> List(Value('string'))


In [8]:
tags = panx_ch["de"]["train"].features["ner_tags"].feature
print(tags)

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])


### 3.3 Readable Tags

Integer ids are hard to read. The function `create_tag_names` turns each list of ids into a list of names. It is used with `Dataset.map`, which adds the new column `ner_tags_str` to every split. The table below the next cell shows the words in the first row and their tags in the second row, so we can check the labels by eye.

In [9]:
def create_tag_names(batch):
    """Convert integer NER tags into readable tag names.

    Args:
        batch: Batch of examples (a dictionary of lists) that contains the
            `ner_tags` column with integer class ids.

    Returns:
        A dictionary with the new `ner_tags_str` column. It holds one list of
        tag names (for example `"B-LOC"`) for each example in the batch.

    Notes:
        The mapping from id to name uses the global `tags` ClassLabel feature.
    """
    return {"ner_tags_str": [tags.int2str(idx) for idx in batch["ner_tags"]]}

In [10]:
# Add a column with readable tag names to every split of the German corpus
panx_de = panx_ch["de"].map(create_tag_names)
de_example = panx_de["train"][0]
# Show one example as a table: words in the first row, tags in the second row
pd.DataFrame([de_example["tokens"], de_example["ner_tags_str"]], ["Tokens", "Tags"])

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


### 3.4 Entity Frequency per Split

Before training, it is good to know how often each entity type appears. The code counts **only `B-` tags**. Each entity has exactly one `B-` tag, so this gives the number of entities and not the number of words inside entities.

The table shows that the three types appear in similar numbers in every split. The validation and test splits each have about half as many entities as the training split. This means that no single class dominates the entity labels in the German corpus. Note that this count does not include the `O` class, which is much larger.

In [11]:
from collections import Counter

split2freqs = defaultdict(Counter)
for split, dataset in panx_de.items():
    for row in dataset["ner_tags_str"]:
        for tag in row:
            # Every entity has exactly one B- tag, so counting B- tags counts entities
            if tag.startswith("B"):
                tag_type = tag.split("-")[1]
                split2freqs[split][tag_type] += 1


pd.DataFrame.from_dict(split2freqs, orient="index")

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


## Section 4 - Comparing Two Tokenizers

### 4.1 Loading the BERT and XLM-R Tokenizers

A transformer model cannot read raw words. A **tokenizer** first splits the text into **subword tokens**, and each token has an id in the model's vocabulary. The way a tokenizer splits text is important for multilingual work, so we compare two of them:

| | `bert-base-cased` | `xlm-roberta-base` |
|---|---|---|
| Algorithm | WordPiece | SentencePiece |
| Marker | `##` at the start of a continuation piece | `▁` at the start of a word |
| Special tokens | `[CLS]`, `[SEP]` | `<s>`, `</s>` |
| Language scope | English | About 100 languages |

SentencePiece works on the raw text and treats the space as a normal character (`▁`). For this reason, it does not need rules that are specific to one language (Kudo and Richardson, 2018).

In [24]:
from transformers import AutoTokenizer
# Compare an English-only tokenizer (BERT) with a multilingual one (XLM-R)
bert_model_name = "bert-base-cased"
xlmr_model_name = "xlm-roberta-base"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
xlmr_tokenizer = AutoTokenizer.from_pretrained(xlmr_model_name)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### 4.2 Tokenizing One Sentence

We tokenize the same English sentence, `"Jack Sparrow loves New York!"`, with both tokenizers. The method `.tokens()` gives the tokens as strings. The next cell prints the BERT tokens, and a later cell shows both lists together.

In [25]:
text = "Jack Sparrow loves New York!"
bert_tokens = bert_tokenizer(text).tokens()
xlmr_tokens = xlmr_tokenizer(text).tokens()

In [30]:
bert_tokens

['[CLS]', 'Jack', 'Spa', '##rrow', 'loves', 'New', 'York', '!', '[SEP]']

### 4.3 Side-by-Side Comparison

The table below lines up the two results. BERT gives 9 tokens and XLM-R gives 10. Both tokenizers split "Sparrow" into two pieces. XLM-R also splits "loves" into `▁love` and `s`. We see that the special tokens are different: BERT adds `[CLS]` and `[SEP]`, and XLM-R adds `<s>` and `</s>`.

In [32]:
# Put both token lists in one table (the shorter list is filled with None)
pd.DataFrame([bert_tokens, xlmr_tokens], ["BERT", "XLMR"])

,0,1,2,3,4,5,6,7,8,9
BERT,[CLS],Jack,Spa,##rrow,loves,New,York,!,[SEP],None
XLMR,<s>,▁Jack,▁Spar,row,▁love,s,▁New,▁York,!,</s>


### 4.4 Reversing the Tokenization

The marker `▁` shows where a word starts. If we join the tokens and replace `▁` with a space, we get the original sentence back, including the special tokens. This property is useful in the next notebook, because it helps us to see which sub-token is the first one of a word.

In [35]:
# Join the tokens and turn the SentencePiece marker (U+2581) back into a space
"".join(xlmr_tokens).replace(u"\u2581", " ")

'<s> Jack Sparrow loves New York!</s>'

## Conclusion

This notebook prepared the data and the vocabulary knowledge for the rest of the chapter. We selected the PAN-X part of XTREME, built a four-language corpus, studied the tag format and compared two tokenizers.

### Corpus summary

| Language | Training examples |
|---|---|
| German (`de`) | 12,580 |
| French (`fr`) | 4,580 |
| Italian (`it`) | 1,680 |
| English (`en`) | 1,180 |

### Entity counts in the German corpus (number of `B-` tags)

| Split | `LOC` | `ORG` | `PER` |
|---|---|---|---|
| `train` | 6,186 | 5,366 | 5,810 |
| `validation` | 3,172 | 2,683 | 2,893 |
| `test` | 3,180 | 2,573 | 3,071 |

### Tokenizer result for `"Jack Sparrow loves New York!"`

| Tokenizer | Number of tokens | Observation |
|---|---|---|
| BERT (`bert-base-cased`) | 9 | `[CLS]` and `[SEP]` added, "Sparrow" split into `Spa` and `##rrow` |
| XLM-R (`xlm-roberta-base`) | 10 | `<s>` and `</s>` added, "Sparrow" and "loves" each split in two |

### What comes next

A tokenizer can split one word into several tokens, but the dataset has **one label per word**. The next notebook, `tokenizing_text.ipynb`, solves this problem by aligning the word labels with the sub-tokens.

## **Resources & References**

**Official Documentation**

- [Hugging Face Datasets: Process (`map`, `shuffle`, `select`)](https://huggingface.co/docs/datasets/process)
- [Hugging Face Datasets: `google/xtreme` dataset page](https://huggingface.co/datasets/google/xtreme)
- [Hugging Face Transformers: Tokenization algorithms](https://huggingface.co/docs/transformers/tokenizer_summary)
- [Model card: `FacebookAI/xlm-roberta-base`](https://huggingface.co/FacebookAI/xlm-roberta-base)
- [`import-ipynb` on PyPI](https://pypi.org/project/import-ipynb/)

**Key Papers / Research**

- Hu, J., Ruder, S., Siddhant, A., Neubig, G., Firat, O., Johnson, M. (2020). [XTREME: A Massively Multilingual Multi-task Benchmark for Evaluating Cross-lingual Generalization](https://arxiv.org/abs/2003.11080).
- Pan, X., Zhang, B., May, J., Nothman, J., Knight, K., Ji, H. (2017). [Cross-lingual Name Tagging and Linking for 282 Languages](https://aclanthology.org/P17-1178/). ACL 2017.
- Conneau, A. et al. (2019). [Unsupervised Cross-lingual Representation Learning at Scale](https://arxiv.org/abs/1911.02116).
- Devlin, J., Chang, M.-W., Lee, K., Toutanova, K. (2018). [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805).
- Kudo, T., Richardson, J. (2018). [SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing](https://arxiv.org/abs/1808.06226).

**Additional Reading**

- Tunstall, L., von Werra, L., Wolf, T. (2022). [Natural Language Processing with Transformers, Revised Edition](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/). O'Reilly Media. Chapter 4: Multilingual Named Entity Recognition.